In [2]:
!pip install -q --upgrade pip
!pip install -q scikit-image
!pip install -q fastapi uvicorn python-multipart pyngrok nest_asyncio pillow einops tqdm

In [3]:
import os, sys, shutil, subprocess

BRANCH = "main"
REPO_URL = "https://github.com/chirana07/Diffusion_new_final.git"
REPO_DIR = "/kaggle/working/Diffunet"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR],
    check=True
)

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print("Repo cloned to:", REPO_DIR)
print("Top-level files:", os.listdir(REPO_DIR))

Cloning into '/kaggle/working/Diffunet'...


Repo cloned to: /kaggle/working/Diffunet
Top-level files: ['model.py', '.gitattributes', 'dataset.py', 'eval_metrics.txt', 'test.py', 'checkpoints', 'evaluation.py', 'diffusion.py', 'modules.py', 'sanity_test.py', 'inference.py', '.git', 'Model', '.gitignore', 'config.py', 'train.py']


In [ ]:
!find /kaggle/working/Diffunet -type f -name "*.pth" | sort

In [9]:
%%writefile /kaggle/working/Diffunet/api.py
import io
import os
import time
import uuid
import base64
import numpy as np
import torch
import torchvision.transforms.functional as TF
from PIL import Image
from pydantic import BaseModel
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn

from config import Config
from model import ResidualConditionedUNet
from diffusion import DiffusionEngine


app = FastAPI(title="Diffunet Kaggle API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

conf = Config()
device = conf.DEVICE

FAST_STEPS = 100
QUALITY_STEPS = 100

CHECKPOINT_PATH = "/kaggle/working/Diffunet/checkpoints/last_pth_only/final.pth"

SESSIONS = {}


class QualityRequest(BaseModel):
    session_id: str


def pad_to_multiple_of_8(img):
    w, h = img.size
    new_w = ((w + 7) // 8) * 8
    new_h = ((h + 7) // 8) * 8
    pad_w = new_w - w
    pad_h = new_h - h
    img = TF.pad(img, [0, 0, pad_w, pad_h], fill=0)
    return img, (w, h)


def tensor_to_pil(t):
    t = (t + 1.0) / 2.0
    t = torch.clamp(t, 0.0, 1.0)
    return TF.to_pil_image(t.squeeze(0).cpu())


def load_model():
    if not os.path.exists(CHECKPOINT_PATH):
        raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

    model = ResidualConditionedUNet().to(device)
    diff = DiffusionEngine()

    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

    if isinstance(checkpoint, dict) and "ema" in checkpoint:
        print("Loading EMA weights...")
        model.load_state_dict(checkpoint["ema"])
    elif isinstance(checkpoint, dict) and "model" in checkpoint:
        print("Loading standard model weights...")
        model.load_state_dict(checkpoint["model"])
    else:
        model.load_state_dict(checkpoint)

    model.eval()
    return model, diff


model, diff = load_model()


def run_sample(low_tensor, inference_steps):
    return diff.sample(model, low_tensor, inference_steps=inference_steps)


def enhance_image_bytes(img_bytes: bytes, inference_steps: int):
    start = time.time()

    try:
        low = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Invalid image: {e}")

    low_padded, original_size = pad_to_multiple_of_8(low)

    low_tensor = (TF.to_tensor(low_padded) - 0.5) * 2.0
    low_tensor = low_tensor.unsqueeze(0).to(device)

    with torch.inference_mode():
        gen_tensor = run_sample(low_tensor, inference_steps)

    gen_img = tensor_to_pil(gen_tensor)
    gen_img = gen_img.crop((0, 0, original_size[0], original_size[1]))

    out_buf = io.BytesIO()
    gen_img.save(out_buf, format="PNG")
    enhanced_bytes = out_buf.getvalue()

    low_np = np.array(low)
    gen_np = np.array(gen_img)

    p = round(float(psnr_fn(low_np, gen_np, data_range=255)), 2)
    s = round(float(ssim_fn(low_np, gen_np, channel_axis=2, data_range=255)), 4)

    elapsed = round(time.time() - start, 2)
    return enhanced_bytes, p, s, elapsed


@app.get("/")
def root():
    return {"status": "ok", "gpu": device}


@app.get("/health")
def health():
    return {
        "status": "ok",
        "gpu": device,
        "fast_steps": FAST_STEPS,
        "quality_steps": QUALITY_STEPS,
    }


@app.post("/api/enhance/fast")
async def enhance_fast(image: UploadFile = File(...)):
    allowed = {"image/jpeg", "image/jpg", "image/png"}
    if image.content_type not in allowed:
        raise HTTPException(status_code=400, detail="Only JPG / PNG allowed.")

    contents = await image.read()
    if not contents:
        raise HTTPException(status_code=400, detail="Empty file.")

    session_id = str(uuid.uuid4())
    SESSIONS[session_id] = contents

    enhanced_bytes, p, s, elapsed = enhance_image_bytes(contents, FAST_STEPS)

    return {
        "success": True,
        "session_id": session_id,
        "original_url": f"data:image/png;base64,{base64.b64encode(contents).decode()}",
        "image_url": f"data:image/png;base64,{base64.b64encode(enhanced_bytes).decode()}",
        "psnr": p,
        "ssim": s,
        "mode": "fast",
        "processing_time": elapsed,
    }


@app.post("/api/enhance/quality")
async def enhance_quality(req: QualityRequest):
    contents = SESSIONS.get(req.session_id)
    if contents is None:
        raise HTTPException(status_code=404, detail="Session not found. Please upload again.")

    enhanced_bytes, p, s, elapsed = enhance_image_bytes(contents, QUALITY_STEPS)

    return {
        "success": True,
        "session_id": req.session_id,
        "image_url": f"data:image/png;base64,{base64.b64encode(enhanced_bytes).decode()}",
        "psnr": p,
        "ssim": s,
        "mode": "quality",
        "processing_time": elapsed,
    }

Overwriting /kaggle/working/Diffunet/api.py


In [10]:
from api import app

for route in app.routes:
    print(getattr(route, "methods", None), getattr(route, "path", None))

{'GET', 'HEAD'} /openapi.json
{'GET', 'HEAD'} /docs
{'GET', 'HEAD'} /docs/oauth2-redirect
{'GET', 'HEAD'} /redoc
{'GET'} /
{'GET'} /health
{'POST'} /api/enhance/fast
{'POST'} /api/enhance/quality


In [ ]:
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok
import nest_asyncio
import uvicorn
from api import app

user_secrets = UserSecretsClient()
NGROK_AUTHTOKEN = user_secrets.get_secret("NGROK_AUTHTOKEN")

NGROK_DOMAIN = None
# NGROK_DOMAIN = "your-domain.ngrok-free.app"

nest_asyncio.apply()

try:
    ngrok.kill()
except Exception:
    pass

ngrok.set_auth_token(NGROK_AUTHTOKEN)

if NGROK_DOMAIN:
    public_tunnel = ngrok.connect(addr=8000, proto="http", domain=NGROK_DOMAIN)
else:
    public_tunnel = ngrok.connect(addr=8000, proto="http")

print("=" * 70)
print("Public URL:", public_tunnel.public_url)
print("Fast:", f"{public_tunnel.public_url}/api/enhance/fast")
print("Quality:", f"{public_tunnel.public_url}/api/enhance/quality")
print("Health:", f"{public_tunnel.public_url}/health")
print("=" * 70)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

Public URL: https://gleamingly-limitary-georgie.ngrok-free.dev
Fast: https://gleamingly-limitary-georgie.ngrok-free.dev/api/enhance/fast
Quality: https://gleamingly-limitary-georgie.ngrok-free.dev/api/enhance/quality
Health: https://gleamingly-limitary-georgie.ngrok-free.dev/health


INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2402:d000:810c:37b6:412b:68b5:2038:58e9:0 - "POST /api/enhance/fast HTTP/1.1" 200 OK
INFO:     2402:d000:810c:37b6:412b:68b5:2038:58e9:0 - "POST /api/enhance/quality HTTP/1.1" 200 OK
INFO:     2402:d000:810c:37b6:412b:68b5:2038:58e9:0 - "POST /api/enhance/fast HTTP/1.1" 200 OK
INFO:     2402:d000:810c:37b6:412b:68b5:2038:58e9:0 - "POST /api/enhance/fast HTTP/1.1" 200 OK


In [7]:
import requests

url = "https://gleamingly-limitary-georgie.ngrok-free.dev/"
r = requests.get(url, headers={"ngrok-skip-browser-warning": "true"}, timeout=30)

print(r.status_code)
print(r.json())

404


JSONDecodeError: Expecting value: line 1 column 1 (char 0)